In [65]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, PCA
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline, PipelineModel

In [66]:
spark = SparkSession.builder \
    .appName("Segmentacion_Perfilado_Clientes") \
    .getOrCreate()

clientes_df = spark.read.parquet("/home/jovyan/work/data/CLIENTS", header=True, inferSchema=True)
behavioural_df = spark.read.parquet("/home/jovyan/work/data/BEHAVIOURAL", header=True, inferSchema=True)

In [67]:
df_id_unido = clientes_df.alias("c").join(
    behavioural_df.alias("b"),
    on="CLIENT_ID",
    how="inner"
)

In [68]:
df_id_unido.show(1)

+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------+------------+-----------------+-------------+--------------+-----------+-----------------+-------+-----------+----------------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+------------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+----------------

In [69]:
len(df_id_unido.columns)

58

In [70]:
NUMERICAL_COLS = [
    "JOB_SENIORITY",
    "INSTALLMENT",
    "FAMILY_SIZE",
    "PROACTIVE_SCORING",
    "BEHAVIORAL_SCORING",
    "DAYS_LAST_INFO_CHANGE",
    "NUMBER_OF_PRODUCTS",
    "TOTAL_INCOME",
    "AMOUNT_PRODUCT",
    "INSTALLMENT",
    "REGION_SCORE",
]

CATEGORICAL_COLS = [
]

In [71]:
df_imputado = df_id_unido.na.fill(0, NUMERICAL_COLS)
df_imputado = df_imputado.na.fill("NA_MISSING", subset=CATEGORICAL_COLS)

In [72]:
df_imputado.show(5)

+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------+------------+-----------------+-------------+--------------+-----------+-----------------+-------+-----------+----------------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+------------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+----------------

In [73]:
#Todas las columnas ya están limpias y sin valores nulos

all_columns = NUMERICAL_COLS + CATEGORICAL_COLS

In [74]:
len(all_columns)

11

In [75]:
indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_Index", handleInvalid="keep")
    for c in CATEGORICAL_COLS
]

encoders = [
    OneHotEncoder(inputCols=[c + "_Index"], outputCols=[c + "_OHE"], dropLast=True)
    for c in CATEGORICAL_COLS
]

In [76]:
indexers

[]

In [77]:
assembler = VectorAssembler(
    inputCols=all_columns,
    outputCol="unscaled_features" # Nombre temporal antes de escalar
)


scaler = StandardScaler(
    inputCol="unscaled_features",
    outputCol="features", # ESTA es tu columna final
    withStd=True,
    withMean=False # No centrar para datos dispersos (OHE), si tienes muchos zeros
)

In [78]:
PCA_COMPONENTS = 10
pca = PCA(k=PCA_COMPONENTS, inputCol="features", outputCol="pca_features")

In [79]:
pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler, pca])

pipeline_model = pipeline.fit(df_imputado)

df_features = pipeline_model.transform(df_imputado)

df_features.select("CLIENT_ID", "features").show(5, False)

print(f"Dimensiones totales del vector 'features': {len(all_columns)}")

+------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|CLIENT_ID   |features                                                                                                                                                                                                         |
+------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|ES182147947X|[0.29233387252426585,2.614051124979187,2.281310270135329,3.8095503356063487,0.0,2.5873136648080646,0.0,1.6056954949819635,2.7444960406161614,2.614051124979187,2.7595859054723357]                               |
|ES182389511E|[0.05498566749779831,1.0976702460102052,2.281310270135329,3.735953097072094,1.99944595